In [ ]:
import os
import torch as tr
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from pathlib import Path

from vre.representations import build_representations_from_cfg
from vre.utils import get_project_root, vre_yaml_load, collage_fn, image_write
from vre_video import VREVideo
from vre_repository import get_vre_repository
from vre_repository.optical_flow.raft import FlowRaft
from vre_repository.depth.dpt import DepthDpt
from vre_repository.depth.marigold import Marigold
from vre_repository.normals.depth_svd import DepthNormalsSVD

device = "cuda" if tr.cuda.is_available() else "cpu"
os.environ["VRE_LOGLEVEL"] = "3"

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
class Ctx:
    def __init__(self, _repr):
        self.repr = _repr
    def __enter__(self):
        self.repr.vre_setup() if self.repr.setup_called is False else None
    def __exit__(self, type, value, traceback):
        self.repr.vre_free()

In [ ]:
# video_path = get_project_root() / "resources/test_video.mp4"
video_path = "/home/mihai/code/ml/video-representations-extractor/examples/vre_manual_representation_run/politehnica_DJI_0741_a2_540p.mp4"
video = VREVideo(video_path)
print(video.shape, video.fps)
h, w = video.shape[1:3]

In [ ]:
os.environ["VRE_DEVICE"] = device = "cuda" if tr.cuda.is_available() else "cpu"
all_representations_dict = vre_yaml_load(Path.cwd() / "cfg.yaml")
device = "cuda" if tr.cuda.is_available() else "cpu"
representations = build_representations_from_cfg(all_representations_dict, representation_types=get_vre_repository())
name_to_repr = {r.name: r for r in representations}
print(representations)

### Safe landing: derived from depth and normals

In [ ]:
from vre.representations import TaskMapper, NpIORepresentation, Representation
from vre.utils import ReprOut, colorize_semantic_segmentation, DiskData, MemoryData, semantic_mapper
from vre_repository.depth import DepthRepresentation
from vre_repository.normals import NormalsRepresentation
from overrides import overrides

class BinaryMapper(TaskMapper, NpIORepresentation):
    """
    Note for future self: this is never generic enough to be in VRE -- we'll keep it in this separate code only
    TaskMapper is the only high level interface that makes sense, so we should focus on keeping that generic and easy.
    """
    def __init__(self, name: str, dependencies: list[Representation], mapping: list[dict[str, list]],
                 mode: str, load_mode: str = "binary"):
        TaskMapper.__init__(self, name=name, dependencies=dependencies, n_channels=2)
        NpIORepresentation.__init__(self)
        assert mode in ("all_agree", "at_least_one", "majority"), (name, mode)
        assert load_mode in ("one_hot", "binary"), (name, load_mode)
        assert len(mapping[0]) == 2, (name, mapping)
        assert len(mapping) == len(dependencies), (name, len(mapping), len(dependencies))
        assert all(mapping[0].keys() == m.keys() for m in mapping), (name, [m.keys() for m in mapping])
        self.original_classes: list[list[str]] = [dep.classes for dep in dependencies]
        self.mapping = mapping
        self.mode = mode
        self.load_mode = load_mode
        self.classes = list(mapping[0].keys())
        self.n_classes = len(self.classes)
        self.color_map = [[0, 0, 0], [255, 255, 255]]
        self.output_dtype = "bool"

    @overrides
    def make_images(self, data: ReprOut) -> np.ndarray:
        x = data.output.argmax(-1) if self.load_mode == "one_hot" else (data.output > 0.5).astype(int)
        x = x[..., 0] if x.shape[-1] == 1 else x
        return colorize_semantic_segmentation(x, self.classes, self.color_map)

    @overrides
    def disk_to_memory_fmt(self, disk_data: DiskData) -> MemoryData:
        assert len(disk_data.shape) == 2 and disk_data.dtype == bool, f"{self.name}: {lo(disk_data)}"
        y = np.eye(2)[disk_data.astype(int)] if self.load_mode == "one_hot" else disk_data
        return MemoryData(y.astype(np.float32))

    @overrides
    def memory_to_disk_fmt(self, memory_data: MemoryData) -> DiskData:
        return memory_data.argmax(-1).astype(bool) if self.load_mode == "one_hot" else memory_data.astype(bool)

    @overrides
    def merge_fn(self, dep_data: list[MemoryData]) -> MemoryData:
        dep_data_argmaxed = [data.argmax(-1) for data in dep_data]
        dep_data_converted = [semantic_mapper(x, mapping, oc)
                              for x, mapping, oc in zip(dep_data_argmaxed, self.mapping, self.original_classes)]

        if self.mode == "all_agree":
            res_argmax = sum(dep_data_converted) == len(self.dependencies)
        elif self.mode == "at_least_one":
            res_argmax = sum(dep_data_converted) > 0
        else:
            res_argmax = sum(dep_data_converted) > len(dep_data_converted) // 2
        return self.disk_to_memory_fmt(res_argmax)


class SafeLandingAreas(BinaryMapper):
    def __init__(self, name: str, depth: DepthRepresentation, camera_normals: NormalsRepresentation,
                 load_mode: str = "binary"):
        dependencies = [depth, camera_normals]
        TaskMapper.__init__(self, name, dependencies=dependencies, n_channels=2)
        self.color_map = [[255, 0, 0], [0, 255, 0]]
        self.classes = ["unsafe-landing", "safe-landing"]
        self.load_mode = load_mode
        self.output_dtype = "bool"

    @overrides
    def merge_fn(self, dep_data: list[MemoryData]) -> MemoryData:
        depth, normals = dep_data[0] if len(dep_data[0].shape) == 2 else dep_data[0][..., 0], dep_data[1]
        v1, v2, v3 = normals.transpose(2, 0, 1)
        where_safe = (v2 > 0.8) * ((v1 + v3) < 1.2) * (depth <= 0.9)
        return self.disk_to_memory_fmt(where_safe)

## Run the representations for two particular frame

In [ ]:
# inference setup (this is done inside VRE's main loop at run() as well)
depth: Marigold = name_to_repr["depth_marigold"]
normals: DepthNormalsSVD = name_to_repr["normals_svd(depth_marigold)"]
safe_landing = SafeLandingAreas("safe-landing-no-sseg", depth, normals)

depth2: DepthDpt = name_to_repr["depth_dpt"]
normals2: DepthNormalsSVD = name_to_repr["normals_svd(depth_dpt)"]
safe_landing2 = SafeLandingAreas("safe-landing-no-sseg", depth2, normals2)

# np.random.seed(43)
mb = 1
ixs = sorted([np.random.randint(0, len(video) - 1) for _ in range(mb)])
# plt.imshow(video[ixs[0]])
# plt.show()

with Ctx(depth):
    y_depth_img = depth.make_images(out_depth := depth.resize(depth.compute(video, ixs), (h, w)))
y_normals_img = normals.make_images(out_normals := normals.compute(video, ixs, [out_depth]))
y_safe_landing_img = safe_landing.make_images(safe_landing.compute(video, ixs, [out_depth, out_normals]))

with Ctx(depth2):
    y_depth2_img = depth2.make_images(out_depth2 := depth2.resize(depth2.compute(video, ixs), (h, w)))
y_normals2_img = normals2.make_images(out_normals2 := normals2.compute(video, ixs, [out_depth2]))
y_safe_landing2_img = safe_landing2.make_images(safe_landing2.compute(video, ixs, [out_depth2, out_normals2]))

for i in range(mb):
    titles = ["RGB", "Depth Marigold", "Normals SVD (Depth Marigold)", "Safe Landing (Depth Marigold)",
              "", "Depth DPT", "Normals SVD (Depth DPT)", "Safe Landing (Depth DPT)"]
    imgs = [out_depth.frames[i], y_depth_img[i], y_normals_img[i], y_safe_landing_img[i],
            y_depth_img[i]*0, y_depth2_img[i], y_normals2_img[i], y_safe_landing2_img[i]]
    img = collage_fn(imgs, titles=titles, rows_cols=(2, 4))
    image_write(img, f"{Path(video_path).stem}_{ixs[i]}.png")
    plt.imshow(img)
    plt.show()


### Optical flow +/-1

In [ ]:
h, w = video.shape[1:3]
# h, w = [540, 960]
print(h, w)
flow = FlowRaft(name="flow_raft", dependencies=[], inference_width=w, inference_height=h, iters=5, small=False, delta=5)
flow_l = FlowRaft(name="flow_raft", dependencies=[], inference_width=w, inference_height=h, iters=5,
                small=False, delta=-5)
flow.device = flow_l.device = device

# np.random.seed(43)
mb = 2
ixs = sorted([np.random.randint(0, len(video) - 1) for _ in range(mb)])
print(ixs)

with Ctx(flow):
    y_flow = flow.compute(video, ixs)
with Ctx(flow_l):
    y_flow_l = flow_l.compute(video, ixs)
print(y_flow.output.reshape(-1, 2).mean(0) * [h, w], y_flow.output.reshape(-1, 2).std(0))
print(y_flow_l.output.reshape(-1, 2).mean(0)  * [h, w], y_flow_l.output.reshape(-1, 2).std(0))
flow_img = flow.make_images(y_flow)
flow_l_img = flow_l.make_images(y_flow_l)
for i in range(mb):
    fig, ax = plt.subplots(1, 3, figsize=(20, 10))
    ax[0].imshow(video[ixs[i]])
    ax[1].imshow(flow_img[i])
    ax[2].imshow(flow_l_img[i])
plt.show()
